### Easy:

#### Preconditions:

In [1]:
import pandas as pd

def load_dataset(file_path):
    try:
        df = pd.read_csv(file_path)
        print("Dataset Loaded:", df.shape)
        print("Columns:", df.columns)
        return df
    except Exception as e:
        print("Error:", e)

df = load_dataset("shopsense_reviews.csv")
df.head()

Dataset Loaded: (10199, 20)
Columns: Index(['review_id', 'customer_id', 'product_id', 'category', 'subcategory',
       'review_text', 'rating', 'sentiment_label', 'review_date',
       'helpful_votes', 'verified_purchase', 'language', 'word_count',
       'has_image', 'device_type', 'reviewer_city', 'product_price',
       'seller_rating', 'delivery_days', 'return_flag'],
      dtype='object')


,review_id,customer_id,product_id,category,subcategory,review_text,rating,sentiment_label,review_date,helpful_votes,verified_purchase,language,word_count,has_image,device_type,reviewer_city,product_price,seller_rating,delivery_days,return_flag
0,R000001,C045992,PCLO0017,Clothing,Sarees,<p>DO NOT BUY THIS</p>. Fake product. Nothing ...,1,Negative,2025-06-23,46.0,True,English,14.0,False,desktop_web,Ahmedabad,22570.73,2.9,NaN,1
1,R000002,C048798,PHOM0081,Home,Storage,Waste of money!!! Too many issues with t...,1,Negative,2025-12-10,42.0,True,Hindi,13.0,True,desktop_web,DELHI,11800.29,3.0,8.0,1
2,R000003,C024089,PCLO0017,Clothing,Sarees,<p>DO NOT BUY THIS</p>. Fake product. Nothing ...,1,Negative,2026-02-24,35.0,True,English,14.0,False,mobile_android,Bnglr,22570.73,4.7,3.0,0
3,R000004,C009298,PELE0031,Electronics,Smartphones,Loved it!! Perfect for daily use. My whole fam...,4,Positive,2025-07-09,36.0,True,Hindi,12.0,False,mobile_ios,DELHI,15266.97,2.8,10.0,0
4,R000005,C049476,PBOO0024,Books,Technical,Amazing value for money. The technical exceede...,5,Positive,2025-09-07,14.0,True,Code-mixed,13.0,True,mobile_android,Hyderabad,18689.07,1.5,6.0,0


In [2]:
import re

def clean_text(text):
    if pd.isnull(text):
        return ""
    
    text = re.sub(r'<.*?>', '', text)            # remove HTML
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)   # remove special chars
    text = text.lower()                          # lowercase
    
    return text

def preprocess_data(df):
    df['clean_review'] = df['review_text'].apply(clean_text)
    return df

df = preprocess_data(df)

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def prepare_data(df):
    X = df['clean_review']
    y = df['sentiment_label']
    
    return train_test_split(X, y, test_size=0.2, random_state=42)

def train_logistic_model(X_train, y_train):
    vectorizer = TfidfVectorizer(max_features=5000)
    X_train_vec = vectorizer.fit_transform(X_train)

    model = LogisticRegression(max_iter=200)
    model.fit(X_train_vec, y_train)

    return model, vectorizer, X_train_vec

X_train, X_test, y_train, y_test = prepare_data(df)
model1, vectorizer, X_train_vec = train_logistic_model(X_train, y_train)
X_test_vec = vectorizer.transform(X_test)

In [4]:
import sklearn
import numpy as np

print("Pandas:", pd.__version__)
print("Sklearn:", sklearn.__version__)
print("NumPy:", np.__version__)

Pandas: 2.3.3
Sklearn: 1.8.0
NumPy: 2.3.0


#### 1. Class Distribution:

In [5]:
def get_class_distribution(df):
    counts = df['sentiment_label'].value_counts()
    percentages = df['sentiment_label'].value_counts(normalize=True) * 100
    
    print("Counts:\n", counts)
    print("\nPercentages:\n", percentages)
    
    return counts, percentages

counts, percentages = get_class_distribution(df)

Counts:
 sentiment_label
Positive    7105
Negative    2095
Neutral      999
Name: count, dtype: int64

Percentages:
 sentiment_label
Positive    69.663693
Negative    20.541230
Neutral      9.795078
Name: proportion, dtype: float64


#### Observed Class Distribution
##### From the dataset:
- Positive: 7105 (69.66%)
- Negative: 2095 (20.54%)
- Neutral: 999 (9.79%)
##### Interpretation:
- The dataset is moderately imbalanced
- Majority class = Positive (~70%)
- Minority classes:
    - Negative (~20%)
    - Neutral (~10%)
##### Why Accuracy is NOT Reliable:
- A model predicting only "Positive" will achieve:
      - ~70% accuracy
- But it will:
      - Miss all negative reviews (~20%)
      - Ignore customer complaints completely

#### 2. Model Evaluation:

In [6]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_model(model, X_test_vec, y_test):
    y_pred = model.predict(X_test_vec)

    print("Classification Report:\n")
    print(classification_report(y_test, y_pred))

    print("\nConfusion Matrix:\n")
    print(confusion_matrix(y_test, y_pred))

    return y_pred

y_pred1 = evaluate_model(model1, X_test_vec, y_test)

Classification Report:

              precision    recall  f1-score   support

    Negative       1.00      0.82      0.90       439
     Neutral       1.00      0.85      0.92       194
    Positive       0.93      1.00      0.96      1407

    accuracy                           0.95      2040
   macro avg       0.98      0.89      0.93      2040
weighted avg       0.95      0.95      0.95      2040


Confusion Matrix:

[[ 361    0   78]
 [   0  164   30]
 [   0    0 1407]]


#### Performance Summary:
- The model performs well overall, with about 95% accuracy, which means most reviews are classified correctly.
- It is especially good at identifying positive reviews, where it correctly captures almost all happy customers.

- For negative reviews (customer complaints), the model correctly identifies most of them, but misses around 18%. This means some unhappy customers may not be detected.

- For neutral reviews, the performance is also good, with only a few misclassifications.

- Business Meaning
      - The system is reliable for understanding overall customer sentiment
      - However, some complaints may go unnoticed, which could impact customer satisfaction

- Conclusion:
      - The model is strong overall, but improving its ability to detect all negative reviews will make it more useful for the business.

### Medium:

#### 3. Model Comparison:

In [7]:
from sklearn.naive_bayes import MultinomialNB

def train_naive_bayes(X_train_vec, y_train):
    model = MultinomialNB()
    model.fit(X_train_vec, y_train)
    return model

model2 = train_naive_bayes(X_train_vec, y_train)

In [8]:
from sklearn.metrics import f1_score
import time

def measure_latency(model, X_sample):
    start = time.time()
    model.predict(X_sample)
    end = time.time()
    return (end - start) * 1000

def compare_models(model1, model2, X_test_vec, y_test):
    y_pred1 = model1.predict(X_test_vec)
    y_pred2 = model2.predict(X_test_vec)

    f1_1 = f1_score(y_test, y_pred1, average='macro')
    f1_2 = f1_score(y_test, y_pred2, average='macro')

    latency1 = measure_latency(model1, X_test_vec[:100])
    latency2 = measure_latency(model2, X_test_vec[:100])

    print("Logistic Regression → F1:", f1_1, "| Latency:", latency1, "ms")
    print("Naive Bayes → F1:", f1_2, "| Latency:", latency2, "ms")

    return y_pred1, y_pred2

y_pred1, y_pred2 = compare_models(model1, model2, X_test_vec, y_test)

Logistic Regression → F1: 0.9272467105640508 | Latency: 0.0 ms
Naive Bayes → F1: 0.9226452408683304 | Latency: 0.0 ms


In [9]:
def evaluate_hinglish(df, model, vectorizer):
    hinglish_df = df[df['language'] != 'en']
    
    if len(hinglish_df) == 0:
        print("No Hinglish data found")
        return
    
    X = hinglish_df['clean_review']
    y = hinglish_df['sentiment_label']
    
    X_vec = vectorizer.transform(X)
    y_pred = model.predict(X_vec)
    
    from sklearn.metrics import f1_score
    score = f1_score(y, y_pred, average='macro')
    
    print("Hinglish F1-score:", score)

evaluate_hinglish(df, model1, vectorizer)

Hinglish F1-score: 0.9281743820638783


#### Model Comparison Results
- Logistic Regression F1-score: 0.927
- Naive Bayes F1-score: 0.923

- Both models perform almost equally well, but Logistic Regression is slightly better

- Speed (Latency)
     - Both models take ~0 ms per prediction
     - This means both are very fast and easily meet the requirement of under 20 ms
- Hinglish Performance
     - Hinglish F1-score: 0.928
- The model performs consistently well even on mixed-language reviews, which is a good sign

- Business Interpretation
     - Both models are fast and reliable
     - Logistic Regression gives slightly better overall accuracy
     - The model also works well for Hinglish reviews (~15% of data)

- Conclusion:

Both models meet the speed requirement and perform well. Logistic Regression performs slightly better overall, so it is the preferred choice. The model also handles mixed-language (Hinglish) reviews effectively, which is important for real-world usage.

#### 4. Cost Model

In [12]:
from sklearn.metrics import confusion_matrix
import numpy as np

COST_FN = 500
COST_FP = 50

def calculate_cost(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    
    total_fn = 0
    total_fp = 0
    
    # Loop through each class
    for i in range(len(cm)):
        fn = np.sum(cm[i, :]) - cm[i, i]   # row sum - diagonal
        fp = np.sum(cm[:, i]) - cm[i, i]   # column sum - diagonal
        
        total_fn += fn
        total_fp += fp
    
    total_cost = (total_fn * COST_FN) + (total_fp * COST_FP)
    
    print("Total False Negatives:", total_fn)
    print("Total False Positives:", total_fp)
    print("Total Cost:", total_cost)
    
    return total_cost

# Run
cost1 = calculate_cost(y_test, y_pred1)
cost2 = calculate_cost(y_test, y_pred2)

Total False Negatives: 108
Total False Positives: 108
Total Cost: 59400
Total False Negatives: 118
Total False Positives: 118
Total Cost: 64900


##### Cost Model & Business Impact
- False Negative (₹500): Missed complaint → high business loss
- False Positive (₹50): Wrong flag → minor issue
- Missing complaints is much more costly

- Model Cost
     - Logistic Regression: ₹59,400
     - Naive Bayes: ₹64,900
- Daily Impact (100k reviews)
     - Logistic Regression: ~₹29 lakh/day
     - Naive Bayes: ~₹32 lakh/day
- Recommendation: 
     - Logistic Regression is preferred because it results in lower business cost, even though both models have similar performance.

#### 5. Technical Brief:

- Recommendation (What Ships & Why)
    - Model: Logistic Regression (TF-IDF)

- Why:
    - Lowest business cost (~₹29 lakh/day vs ₹32 lakh)
    - High overall performance (F1 ≈ 0.93)
    - Meets latency requirement (<20 ms)
    - Works well on both English and Hinglish reviews

- Limitations:
    - May miss some negative reviews (~18%)
    - Performance can drop if new types of reviews appear

- Production Monitoring Plan

   - Metric to Track (Weekly):
        - Recall for Negative class

   - Threshold for Retraining:
        - If Recall < 70%

-  Detecting Model Degradation

   - Monitor:
        - Increase in positive predictions
        - Drop in negative recall
        - Sudden rise in customer complaints
        
- Conclusion:

The model is suitable for deployment due to lower cost and strong performance, but continuous monitoring is required to ensure it remains reliable over time.